# 1. Finding Slope and Intercept of a Line

Goal of linear regression, find the straight line that gets as close as possible to all the points at the same time. This is a longer topic than we have time for, but let's try to find a line that best fits your data.

In [ ]:
from typing import Callable, Sequence, NamedTuple

import matplotlib.pyplot as plt
import random 
import numpy as np

In [ ]:
LineFunction = Callable[[float | int], float | int]

def make_linear_function(m: float | int, b: float | int, noise: float | int = 0) -> LineFunction:
    def f(x: float | int) -> float | int:
        return m * x + b + random.uniform(-noise, noise)
    return f

In [ ]:
class Point(NamedTuple):
    x: float | int
    y: float | int

def plot_points(points: Sequence[Point]):
    x_values = [point.x for point in points]
    y_values = [point.y for point in points]

    plt.scatter(x_values, y_values, color='blue', label='Data Points')
    
    plt.xlabel('X')
    plt.ylabel('Y')
    plt.title('Data Points')
    plt.legend()
    plt.grid(True, alpha=0.2)
    plt.show()

def plot_points_and_line(points: Sequence[Point], line_function: LineFunction):
    x_values = [point.x for point in points]
    y_values = [point.y for point in points]

    plt.scatter(x_values, y_values, color='blue', label='Data Points')
    
    x_line = np.linspace(min(x_values), max(x_values), 100)
    y_line = [line_function(x) for x in x_line]
    
    plt.plot(x_line, y_line, color='red', label='Line of Best Fit')
    
    plt.xlabel('X')
    plt.ylabel('Y')
    plt.title('Data Points and Line of Best Fit')
    plt.legend()
    plt.grid(True, alpha=0.2)
    plt.show()

## Step 1. Build your data

In [ ]:
# ---INPUT---
# Choose slope, y-intercept, and noise level
# f(x) = mx + b + random.uniform(-noise, noise)

M = 2
B = 1
NOISE = 10

In [ ]:
f = make_linear_function(M, B, NOISE)

POINTS: list[Point] = []
for x in np.linspace(-50, 50, 100):
    y = f(x)
    POINTS.append(Point(x=x, y=y))


In [ ]:
# Make sure your plot resembles a line, otherwise reduce your noise level
plot_points(POINTS)

## Step 2. Visualizing the best fit line

In [ ]:
# ---INPUTS---
# Change your guess for the slope and y-intercept of the line

M_GUESS = 2.5
B_GUESS = 0

plot_points_and_line(POINTS, make_linear_function(M_GUESS, B_GUESS))

## Step 3. Calculating how good our best fit line is

To calculate how good our line is, we will calculate the Sum of Squared Residuals. For every point x, how far (squared) is the actual y value from the simulated y value from our approximate line.

$$SSE = ∑ (yᵢ - ŷᵢ)²$$

![](./static/residuals.png)

In [ ]:
def calculate_sum_squared_error(points: Sequence[Point], line_function_guess: LineFunction) -> float:

    # TODO: Implement this function to calculate the sum of squared errors between the actual y values and the guessed y values from the line function guess
    return 0.0



In [ ]:
calculate_sum_squared_error(POINTS, make_linear_function(M_GUESS, B_GUESS))

In [ ]:
# ---INPUTS---
# Change your guess for the slope and y-intercept of the line
M_GUESS = 2.5
B_GUESS = 0

print (calculate_sum_squared_error(POINTS, make_linear_function(M_GUESS, B_GUESS)))
plot_points_and_line(POINTS, make_linear_function(M_GUESS, B_GUESS))


## Step 4. Brute Force Approach

If we try a bunch of different slope and intercepts, we could track the sum squared error values and see which combination yields the lowest. Let's try it.

In [ ]:
# ---INPUTS---

M_CENTER = 2.5 # Choose a center point for the slope guesses
M_SPREAD = 5   # Choose how far above and below the center point to spread the slope guesses
M_N = 80       # Choose how many slope guesses to generate

B_CENTER = 0
B_SPREAD = 5
B_N = 40

# Generate a list of slope and y-intercept guesses to try out
m_samples = np.linspace(M_CENTER - M_SPREAD, M_CENTER + M_SPREAD, M_N)
b_samples = np.linspace(B_CENTER - B_SPREAD, B_CENTER + B_SPREAD, B_N)

print (m_samples, b_samples)


In [ ]:
# TODO: Create a grid of slope and intercept guesses, calculate the SSE for each guess. Plot the values. Adjust your search guesses center and spread to narrow in on a local minimum


In [ ]:
# Sanity check #1
# For given m and b, does the printed value of the SSE match what is shown on the graph? Are you plotting it correctly?

m = 2
b = 1

calculate_sum_squared_error(POINTS, make_linear_function(m, b))

In [ ]:
# Sanity check #2
# Once you have found a local minimum, plot the line! Does it look reasonable?

m = 2.1
b = 1.7

plot_points_and_line(POINTS, make_linear_function(m, b))

In [ ]:
# Example code to plot a contour plot of the SSE surface for different slope and intercept guesses

slope_samples = [0,1,2,3,4] # 5 slope values
intercept_samples = [0,1,2,3] # 4 intercept values

slope_grid, intercept_grid = np.meshgrid(slope_samples, intercept_samples)

"""
slope_grid = [
    [0, 1, 2, 3, 4],
    [0, 1, 2, 3, 4],
    [0, 1, 2, 3, 4],
    [0, 1, 2, 3, 4]
]

intercept_grid = [
    [0, 0, 0, 0, 0],
    [1, 1, 1, 1, 1],
    [2, 2, 2, 2, 2],
    [3, 3, 3, 3, 3]
]
"""

# These are made up SSE values. Change this code to calculate the actual SSE values for each slope and intercept guess and fill in the sse_grid with those values instead of these made up values.
sse_grid = [[8,7,6,5,4], [5,4,3,2,3], [2,1,0,1,2], [3,2,3,4,5]]

contour = plt.contourf(slope_grid, intercept_grid, sse_grid, levels=30, cmap='viridis')
plt.colorbar(contour, label='SSE')
plt.contour(slope_grid, intercept_grid, sse_grid, levels=10, colors='white', alpha=0.4, linewidths=0.8)

plt.xlabel('Slope (m)')
plt.ylabel('Intercept (b)')
plt.title('SSE Surface Example')
plt.grid(True, alpha=0.2)
plt.legend()


## Step 5. Best fit line - the actual way

#### Ordinary Least Squares

$$m = \frac{n \sum (x_i y_i) - (\sum x_i)(\sum y_i)}{n \sum x_i^2 - (\sum x_i)^2}$$

$$b = \bar{y} - m \cdot \bar{x} \quad \text{   or   } \quad b = \frac{\sum y_i - m \sum x_i}{n}$$

In [ ]:
# TODO: implement a function to calculate the best fit slope and intercept for a given set of points

class BestFitLine(NamedTuple):
    m: float
    b: float

def calculate_best_fit_line(points: Sequence[Point]) -> BestFitLine:
    return BestFitLine(m=0, b=0)



In [ ]:
best_fit_line = calculate_best_fit_line(POINTS)
print (best_fit_line)

In [ ]:
# Sanity Check
plot_points_and_line(POINTS, make_linear_function(best_fit_line.m, best_fit_line.b))

## Step 6. Just use numpy

In [ ]:
m, b  = np.polyfit([p.x for p in POINTS], [p.y for p in POINTS], deg=1)

numpy_best_fit_line = BestFitLine(m=m, b=b)

print ("Slope from np.polyfit:", m)
print ("Intercept from np.polyfit:", b)
print ("SSE from np.polyfit:", calculate_sum_squared_error(POINTS, make_linear_function(m, b)))
print ()
print ("Slope from my best fit line:", best_fit_line.m)
print ("Intercept from my best fit line:", best_fit_line.b)
print ("SSE from my best fit line:", calculate_sum_squared_error(POINTS, make_linear_function(best_fit_line.m, best_fit_line.b)))

plot_points_and_line(POINTS, make_linear_function(m, b))


## Step 7: Working with polynomial functions

Data is rarely linear, let's explore how to model this

In [ ]:
def print_polynomial_function(coefficients: Sequence[float | int]):
    terms = []
    for i, coeff in enumerate(coefficients):
        power = len(coefficients) - 1 - i
        if power == 0:
            terms.append(f"{coeff}")
        elif power == 1:
            terms.append(f"{coeff}x")
        else:
            terms.append(f"{coeff}x^{power}")
    print(" + ".join(terms))

def make_polynomial_function(coefficients: Sequence[float | int]) -> LineFunction:
    def f(x: float | int) -> float | int:
        y = 0
        for i, coeff in enumerate(coefficients):
            y += coeff * (x ** (len(coefficients) - 1 - i))

            # to add noise
            # y *= (1 + random.uniform(-0.1, 0.1))

        return y
    return f



In [ ]:
COEFFICIENTS = [0.01, 0.1, 0.5, 1]
f2 = make_polynomial_function(COEFFICIENTS)

print_polynomial_function(COEFFICIENTS)

POINTS2: list[Point] = []
for x in np.linspace(-50, 50, 100):
    y = f2(x)
    POINTS2.append(Point(x=x, y=y))

plot_points(POINTS2)

In [ ]:
# TODO: try changes the deg parameter, see how the best fit line changes
# TODO: try adding noise, how does that affect the best fit line? Does the line look `OVERFIT`?

best_fit_coefficients = np.polyfit([p.x for p in POINTS2], [p.y for p in POINTS2], deg=3)

print_polynomial_function(list(best_fit_coefficients))
print (calculate_sum_squared_error(POINTS2, make_polynomial_function(list(best_fit_coefficients))))

plot_points_and_line(POINTS2, make_polynomial_function(list(best_fit_coefficients)))


### Other Resources

[Best Fit Line](https://www.youtube.com/watch?v=PaFPbb66DxQ)

[Bias and Variance](https://www.youtube.com/watch?v=EuBBz3bI-aA)